# 2nd Law of Thermodynamics — Teaching Demo

Python translation of `teaching_demo.nb`. We simulate energy diffusion on a 2D lattice using a simple random-walk rule and watch four things:

1. A single quantum doing a 2D random walk (microscopic intuition).
2. A small 4×4 system that already sits at thermal equilibrium — fluctuations only.
3. A larger 10×20 system that starts far from equilibrium and relaxes toward it.
4. The combinatorial count of microstates Ω(N, q) and why the equilibrium $q_A$ is overwhelmingly probable.

The rule: each step, every energy quantum on a non-zero site independently picks one of {stay, up, down, left, right} with equal probability. Quanta that would leave the grid are reflected back (clamped at the boundary). Energy is conserved.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

rng = np.random.default_rng(0)
plt.rcParams['figure.dpi'] = 100


## 1. Random walk on a 2D grid

One quantum starts at the centre of a 20×20 grid. We also keep a *memory* grid that records every site the quantum has ever visited.

In [ ]:
def step(state):
    """One diffusion step. ``state`` is a 2D int array of energy quanta.

    For each quantum on each non-zero site, pick one of {stay, ±x, ±y} uniformly
    and move it. Out-of-bounds moves are clamped to the boundary. Energy is conserved.
    """
    nrows, ncols = state.shape
    new = np.zeros_like(state)
    moves = np.array([[0, 0], [1, 0], [0, 1], [-1, 0], [0, -1]])
    rs, cs = np.nonzero(state)
    for r, c in zip(rs, cs):
        n = int(state[r, c])
        # Vectorise the n moves at this site
        choices = moves[rng.integers(0, 5, size=n)]
        new_r = np.clip(r + choices[:, 0], 0, nrows - 1)
        new_c = np.clip(c + choices[:, 1], 0, ncols - 1)
        np.add.at(new, (new_r, new_c), 1)
    return new


def step_wmem(active, memory):
    """Random-walk step that also marks every visited site in ``memory``."""
    new_active = step(active)
    new_memory = memory.copy()
    new_memory[new_active > 0] = 1
    return new_active, new_memory


In [ ]:
# Initial state: a single quantum at (10, 10) in a 20x20 grid
N = 20
active = np.zeros((N, N), dtype=int)
memory = np.zeros((N, N), dtype=int)
active[10, 10] = 1

actives = [active.copy()]
memories = [memory.copy()]

n_steps = 10000
for _ in range(n_steps):
    active, memory = step_wmem(active, memory)
    actives.append(active.copy())
    memories.append(memory.copy())

print(f"Stored {len(actives)} frames; final visited cells: {memories[-1].sum()} / {N*N}")


### Snapshots — current position (left) and trail of visited sites (right)

In [ ]:
snap_steps = [1, 50, 500, 2000, 10000]
fig, axes = plt.subplots(2, len(snap_steps), figsize=(3 * len(snap_steps), 6))
for j, t in enumerate(snap_steps):
    axes[0, j].imshow(actives[t], cmap='Reds', vmin=0, vmax=1.1)
    axes[0, j].set_title(f"step {t}\nposition")
    axes[0, j].set_xticks([]); axes[0, j].set_yticks([])
    axes[1, j].imshow(memories[t], cmap='Greys', vmin=0, vmax=1.1)
    axes[1, j].set_title("visited sites")
    axes[1, j].set_xticks([]); axes[1, j].set_yticks([])
plt.tight_layout()
plt.show()


### Animation

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(7, 3.5))
imA = axA.imshow(memories[0], cmap='Greys', vmin=0, vmax=1.1)
imB = axB.imshow(actives[0], cmap='Reds',  vmin=0, vmax=1.1)
axA.set_title('visited sites'); axB.set_title('current position')
for ax in (axA, axB):
    ax.set_xticks([]); ax.set_yticks([])
txt = axA.text(0.05, 0.95, '', transform=axA.transAxes, color='red', fontsize=12, va='top')

stride = 50  # show every 50th frame to keep the animation light
frames = range(0, len(actives), stride)

def update(i):
    imA.set_data(memories[i])
    imB.set_data(actives[i])
    txt.set_text(f"step {i}")
    return imA, imB, txt

anim = animation.FuncAnimation(fig, update, frames=frames, interval=60, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


### Histogram of visited x-positions over time

The Mathematica version flattens the 20×20 grid and records the index of the single non-zero cell at each step, then histograms those indices. Over many steps the walker explores the whole grid and the distribution becomes roughly uniform (with the boundaries biased upward because of the reflecting clamp).

In [ ]:
indices = []
for a in actives:
    flat = a.flatten()
    idx = np.argmax(flat)  # exactly one non-zero entry, so argmax = its index
    if flat[idx] != 0:
        indices.append(idx)

plt.figure(figsize=(8, 3))
plt.hist(indices, bins=400, range=(0, 400), color='C1')
plt.xlabel('flattened cell index (0..399)')
plt.ylabel('visits over 10000 steps')
plt.title('Visits per cell over the random walk')
plt.show()


## 2. Small 4×4 system starting at thermal equilibrium

Now distribute several quanta and watch how energy redistributes itself. The grid is 4 rows × 4 columns; we put one quantum at the leftmost and rightmost column of every row. So the left half (cols 0–1) and the right half (cols 2–3) both start with 4 quanta. We define a "temperature" for each side as the total energy on that side.

In [ ]:
rows, cols = 4, 4
state = np.zeros((rows, cols), dtype=int)
state[:, 0] = 1  # leftmost column
state[:, -1] = 1  # rightmost column

states_bar = [state.copy()]
for _ in range(200):
    state = step(state)
    states_bar.append(state.copy())

Tleft  = np.array([s[:, :2].sum() for s in states_bar])
Tright = np.array([s[:, 2:].sum() for s in states_bar])
print(f"Final left/right totals: {Tleft[-1]}, {Tright[-1]} (total = {Tleft[-1] + Tright[-1]})")


In [ ]:
plt.figure(figsize=(6, 3))
plt.hist([Tleft, Tright], bins=range(0, 9), label=['left half', 'right half'],
         color=['C1', 'C0'], alpha=0.6, edgecolor='k')
plt.xlabel('total quanta on side'); plt.ylabel('time steps')
plt.legend(); plt.title('Fluctuations around equilibrium (4×4, 8 quanta)')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(states_bar[0], cmap='hot', vmin=0, vmax=max(s.max() for s in states_bar))
ax.axvline(1.5, color='blue', lw=2)
ax.set_xticks([]); ax.set_yticks([])
left_txt  = ax.text(0.05, 0.95, '', transform=ax.transAxes, color='red', fontsize=14, va='top')
right_txt = ax.text(0.95, 0.95, '', transform=ax.transAxes, color='red', fontsize=14, va='top', ha='right')

def update(i):
    im.set_data(states_bar[i])
    left_txt.set_text(f'T={Tleft[i]}')
    right_txt.set_text(f'T={Tright[i]}')
    return im, left_txt, right_txt

anim = animation.FuncAnimation(fig, update, frames=len(states_bar), interval=80, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


## 3. Larger system: relaxation to thermal equilibrium

A 10×20 grid. The left 2 columns (20 sites, region A) start with 10 quanta each — so $q_A = 200$, very hot. The right 18 columns (180 sites, region B) start empty. We define a "temperature" for each side as **average energy per site**: $T_A = q_A / 20$, $T_B = q_B / 180$. Initially $T_A = 10$, $T_B = 0$. Equilibrium is $T_A = T_B = 200/200 = 1$.

In [ ]:
rows, cols = 10, 20
state = np.zeros((rows, cols), dtype=int)
state[:, 0:2] = 10  # cold start: all energy in the leftmost 2 columns

states_eq = [state.copy()]
n_steps = 2000
for _ in range(n_steps):
    state = step(state)
    states_eq.append(state.copy())

Eleft  = np.array([s[:, :2].sum() for s in states_eq])
Eright = np.array([s[:, 2:].sum() for s in states_eq])
Tleft  = Eleft  / 20.0
Tright = Eright / 180.0
print(f"Total quanta (should stay 200): start={Eleft[0]+Eright[0]}, end={Eleft[-1]+Eright[-1]}")
print(f"Final T_left={Tleft[-1]:.3f}, T_right={Tright[-1]:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
vmax = max(s.max() for s in states_eq[:50])  # initial dynamic range
im = ax.imshow(states_eq[0], cmap='hot', vmin=0, vmax=vmax)
ax.axvline(1.5, color='blue', lw=2)
ax.set_xticks([]); ax.set_yticks([])
qa_txt = ax.text(0.02, 0.95, '', transform=ax.transAxes, color='cyan', fontsize=12, va='top')
qb_txt = ax.text(0.98, 0.95, '', transform=ax.transAxes, color='cyan', fontsize=12, va='top', ha='right')

stride = 20
frames = range(0, len(states_eq), stride)

def update(i):
    im.set_data(states_eq[i])
    qa_txt.set_text(f'q_A={Eleft[i]}')
    qb_txt.set_text(f'q_B={Eright[i]}')
    return im, qa_txt, qb_txt

anim = animation.FuncAnimation(fig, update, frames=frames, interval=60, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


### Energy and temperature over time

The total energies on each side relax monotonically, and the temperatures converge to a common value of 1.0.

In [ ]:
t = np.arange(len(states_eq))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))

ax1.plot(t, Eleft,  label='$q_A$ (left, 20 sites)', color='C3')
ax1.plot(t, Eright, label='$q_B$ (right, 180 sites)', color='C0')
ax1.set_xlabel('step'); ax1.set_ylabel('total quanta'); ax1.legend()
ax1.set_title('Energy on each side')

ax2.plot(t, Tleft,  label='$T_A$', color='C3')
ax2.plot(t, Tright, label='$T_B$', color='C0')
ax2.axhline(1.0, color='k', ls='--', lw=0.7, label='equilibrium T = 1')
ax2.set_xlabel('step'); ax2.set_ylabel('mean quanta per site (T)'); ax2.legend()
ax2.set_title('Temperature on each side')
plt.tight_layout(); plt.show()


## 4. Number of microstates Ω(N, q)

For an Einstein solid with $N$ oscillators sharing $q$ quanta of energy,

$$\Omega(N, q) = \binom{q + N - 1}{q} = \frac{(q + N - 1)!}{q!\,(N-1)!}.$$

For the composite system of part 3 ($N_A = 20$ oscillators, $N_B = 180$ oscillators, $q_{\text{tot}} = 200$ quanta) we plot $\log \Omega_A(q_A) \cdot \Omega_B(q_{\text{tot}} - q_A)$ as a function of $q_A$. The peak gives the overwhelmingly most likely macrostate — that's the equilibrium.

In [ ]:
from math import lgamma

def log_omega(N, q):
    # log of (q+N-1)! / (q! (N-1)!)
    return lgamma(q + N) - lgamma(q + 1) - lgamma(N)

def omega(N, q):
    return int(round(np.exp(log_omega(N, q))))

# Sanity checks reproducing the Mathematica section
print(f"Omega(200, 200)       = {np.exp(log_omega(200, 200)):.3e}")
print(f"Omega(20,21)*Omega(180,179) = {np.exp(log_omega(20, 21) + log_omega(180, 179)):.3e}")
print(f"Omega(20,20)*Omega(180,180) = {np.exp(log_omega(20, 20) + log_omega(180, 180)):.3e}")
print(f"Omega(20,18)*Omega(180,182) = {np.exp(log_omega(20, 18) + log_omega(180, 182)):.3e}")


In [ ]:
qA = np.arange(0, 201)
qB = 200 - qA
logOmegaAB = np.array([log_omega(20, a) + log_omega(180, b) for a, b in zip(qA, qB)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.semilogy(qA, np.exp(logOmegaAB))
ax1.set_xlabel('$q_A$'); ax1.set_ylabel(r"$\Omega_A\,\Omega_B$ (log scale)")
ax1.set_title("Number of microstates of the composite system")

ax2.plot(qA, logOmegaAB)
ax2.set_xlabel('$q_A$'); ax2.set_ylabel(r"$S/k_B = \log(\Omega_A\,\Omega_B)$")
ax2.set_title("Entropy of the composite system")

qA_peak = qA[np.argmax(logOmegaAB)]
ax2.axvline(qA_peak, color='r', ls='--', lw=0.8, label=f'peak at $q_A={qA_peak}$')
ax2.axvline(20, color='gray', ls=':', lw=0.8, label='equipartition $q_A=20$')
ax2.legend()
plt.tight_layout(); plt.show()

print(f"Peak of log(Omega_A * Omega_B) at q_A = {qA_peak}")
print("Naive equipartition (equal mean quanta per oscillator): q_A = 200 * 20/200 = 20")
print("The discrete peak is one quantum off because the (q+N-1)! form isn't symmetric in q,N.")


The peak sits essentially at the equipartition value $q_A \approx 20$ where the mean energy per oscillator is the same on both sides — exactly the equilibrium found by the random-walk simulation in part 3. Same point, two derivations: the macrostate that maximises $S = k_B \log \Omega$ is the one the system spends almost all its time in. That's the **second law** in microstate-counting form.